In [2]:
from datasets.fsmol_dock import FsDockDataset
from datasets.partitioned_fsmol_dock import FsDockDatasetPartitioned
# from visualize import make_fig



In [17]:
import plotly.graph_objects as go

def plotly_edges(graph, start_l, end_l):
    start_pos = graph[start_l].pos
    end_pos = graph[end_l].pos
    edges = graph[start_l, end_l].edge_index
    xe=[]
    ye=[]
    ze=[]
    for s_pos, e_pos in zip(start_pos[edges[0]],end_pos[edges[1]]):
        xe+=[s_pos[0], e_pos[0], None]
        ye+=[s_pos[1], e_pos[1], None]
        ze+=[s_pos[2], e_pos[2], None]
    return {'x': xe, 'y':ye, 'z':ze, 'mode' : 'lines', 'name': f'{start_l}-{end_l}'}

def plotly_nodes(graph,l):
    poses = graph[l].pos
    xn=[]
    yn=[]
    zn=[]
    for pos in poses:
        xn.append(pos[0])
        yn.append(pos[1])
        zn.append(pos[2])
    return {'x': xn, 'y':yn, 'z':zn, 'mode': 'markers', 'name': l}

def plotly_holes(graph):
    poses = graph['ligand'].pos
    xn=[]
    yn=[]
    zn=[]
    for pos in poses[graph.hole_neighbors]:
        xn.append(pos[0])
        yn.append(pos[1])
        zn.append(pos[2])
    return {'x': xn, 'y':yn, 'z':zn, 'mode': 'markers', 'name': 'holes'}

def make_fig(graph) -> go.Figure:
    traces=[
        go.Scatter3d(**plotly_edges(graph, 'ligand', 'ligand'), line=dict(color='red', width=5)),
        go.Scatter3d(**plotly_nodes(graph, 'ligand'), marker=dict(symbol='circle', size=6, color='blue')),
        go.Scatter3d(**plotly_nodes(graph, 'receptor'), marker=dict(symbol='circle', size=3, color='green')),
        go.Scatter3d(**plotly_edges(graph, 'atom', 'receptor'), line=dict(color='yellow', width=3)),
        go.Scatter3d(**plotly_nodes(graph, 'atom'), marker=dict(symbol='circle', size=3, color='orange')),
        go.Scatter3d(**plotly_edges(graph, 'atom', 'atom'), line=dict(color='pink', width=1)),
        go.Scatter3d(**plotly_edges(graph, 'receptor', 'receptor'), line=dict(color='grey', width=3)),
        go.Scatter3d(**plotly_edges(graph, 'ligand', 'atom'), line=dict(color='light blue', width=3)),
        go.Scatter3d(**plotly_edges(graph, 'ligand', 'receptor'), line=dict(color='light green', width=3)),
        go.Scatter3d(**plotly_holes(graph), marker=dict(symbol='circle', size=7, color='black')),
        
    ]
    fig = go.Figure(data=traces, layout=go.Layout(
    width=500,
    height=500,
        ))
    return fig

In [11]:

import torch


def mask_graph_sidechains(graph, molecule_sidechain_mask_idx):
    device = graph['ligand'].x.device
    masks = {
        node_t:(
            (graph.sidechains_mask < molecule_sidechain_mask_idx).to(device)
            if node_t == "ligand"
            else torch.arange(graph[node_t].num_nodes, device=device)
        )
        for node_t in graph.metadata()[0]
    }
    return graph.subgraph(masks)


In [7]:

from datasets.fsmol_dock_clf import FsDockClfDataset

ds = FsDockDatasetPartitioned('data/fsdock/smol','data/fsdock/smol_tasks.csv')
ds.load()
# ds = FsDockClfDataset('data/fsdock/clfs/test','data/fsdock/test_tasks.csv')

[2025-Mar-04 22:45:21 IST] [partitioned_fsmol_dock.py:214] INFO - started load
[2025-Mar-04 22:45:35 IST] [partitioned_fsmol_dock.py:233] INFO - finished load


In [14]:
# from datasets.process_mols import hide_sidechains


iterable = iter(range(100))
graph

HeteroData(
  smiles='C=c1cc2c(=Cc3ccc(SC)cc3)oc(O)c2c(=S)[nH]1',
  core_smiles='[1*]c1ccc(C=c2oc([2*])c3c(=S)[nH]c(=C)cc23)cc1',
  sidechains_smiles='[1*]SC.[2*]O',
  sidechains_mask=[21],
  hole_neighbors=[2],
  activity_type='B',
  label=0,
  task='CHEMBL1000314',
  ligand={
    pos=[21, 3],
    x=[21, 17],
  },
  receptor={
    x=[113, 1281],
    pos=[113, 3],
  },
  atom={
    x=[53, 4],
    pos=[53, 3],
  },
  (ligand, lig_bond, ligand)={
    edge_index=[2, 466],
    edge_attr=[466, 4],
  },
  (receptor, to, receptor)={ edge_index=[2, 1838] },
  (ligand, to, receptor)={ edge_index=[2, 1840] },
  (atom, to, atom)={ edge_index=[2, 574] },
  (ligand, to, atom)={ edge_index=[2, 179] },
  (atom, to, receptor)={ edge_index=[2, 53] }
)

In [18]:
i = next(iterable)
# i=100
graph = ds[i*10]
# print(ds._split_indexes[i])
make_fig(graph).show()

In [10]:
ei = {}
for s,d in zip(*graph['ligand','ligand'].edge_index):
    k = (s.item(),d.item())
    if k in ei:
        ei[k]+=1
    else:
        ei[k]=1
ei
        

{(0, 1): 2,
 (1, 0): 2,
 (1, 2): 2,
 (2, 1): 2,
 (1, 3): 2,
 (3, 1): 2,
 (3, 4): 2,
 (4, 3): 2,
 (3, 5): 2,
 (5, 3): 2,
 (5, 6): 2,
 (6, 5): 2,
 (6, 7): 2,
 (7, 6): 2,
 (7, 8): 2,
 (8, 7): 2,
 (8, 9): 2,
 (9, 8): 2,
 (9, 10): 2,
 (10, 9): 2,
 (10, 11): 2,
 (11, 10): 2,
 (11, 12): 2,
 (12, 11): 2,
 (12, 13): 2,
 (13, 12): 2,
 (13, 14): 2,
 (14, 13): 2,
 (14, 15): 2,
 (15, 14): 2,
 (15, 16): 2,
 (16, 15): 2,
 (16, 17): 2,
 (17, 16): 2,
 (17, 18): 2,
 (18, 17): 2,
 (18, 19): 2,
 (19, 18): 2,
 (9, 20): 2,
 (20, 9): 2,
 (20, 21): 2,
 (21, 20): 2,
 (21, 6): 2,
 (6, 21): 2,
 (18, 10): 2,
 (10, 18): 2,
 (17, 12): 2,
 (12, 17): 2,
 (2, 0): 1,
 (3, 0): 1,
 (4, 0): 1,
 (5, 0): 1,
 (6, 0): 1,
 (7, 0): 1,
 (21, 0): 1,
 (20, 0): 1,
 (9, 0): 1,
 (10, 0): 1,
 (8, 0): 1,
 (19, 0): 1,
 (18, 0): 1,
 (14, 0): 1,
 (15, 0): 1,
 (16, 0): 1,
 (17, 0): 1,
 (13, 0): 1,
 (12, 0): 1,
 (11, 0): 1,
 (4, 1): 1,
 (5, 1): 1,
 (6, 1): 1,
 (7, 1): 1,
 (21, 1): 1,
 (20, 1): 1,
 (9, 1): 1,
 (10, 1): 1,
 (8, 1): 1,
 (19, 1

In [13]:
ea = graph['ligand','ligand'].edge_attr
eas = ea.sum(-1)
eas.sum()


tensor(48.)

In [14]:
from torch_geometric.transforms import ToUndirected

In [20]:
uea = ToUndirected()(graph)['ligand','ligand'].edge_attr
uea.shape

torch.Size([462, 4])

In [17]:
graph['ligand','ligand'].edge_attr.shape

torch.Size([510, 4])

In [21]:
510-48

462